In [ ]:
# ==========================================
# CELL 1: Data Loading, Splitting & Preprocessing (Colab Ready)
# ==========================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. Define 25 Predictors & Target Tier
predictors = [
    'AQI_lag24', 'Raw_Conc_lag24', 'Hour', 'Month', 'DayOfWeek',
    'temperature_2m', 'relativehumidity_2m', 'dewpoint_2m', 'apparent_temperature',
    'precipitation', 'rain', 'surface_pressure', 'cloudcover', 'cloudcover_low',
    'cloudcover_mid', 'cloudcover_high', 'windspeed_10m', 'winddirection_10m',
    'windgusts_10m', 'et0_fao_evapotranspiration', 'vapor_pressure_deficit',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'weathercode'
]
target = 'Hazard_Tier'
seasons = ['winter', 'summer', 'monsoon', 'post_monsoon']

# Dictionary to hold preprocessed data per season
processed_data = {}

print("--- [CELL 1] Loading and Preprocessing Seasonal Datasets ---")

for season in seasons:
    # Look for files directly in Colab's current working directory (/content/)
    possible_names = [f"{season}.csv", f"data_{season}.csv", f"dhaka_{season}.csv"]
    file_path = None

    for name in possible_names:
        if os.path.exists(name):
            file_path = name
            break

    if file_path is None:
        raise FileNotFoundError(f"Could not find CSV for '{season}'. Expected one of: {possible_names}")

    df = pd.read_csv(file_path)

    # Time-based 80/20 train-test split within season
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    X_train_raw = train_df[predictors]
    y_train_raw = train_df[target]
    X_test_raw = test_df[predictors]
    y_test_raw = test_df[target]

    # Scale Features via StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Apply SMOTE to training set for class balance
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train_raw)

    # Store processed splits
    processed_data[season] = {
        'X_train': X_train_res,
        'y_train': y_train_res,
        'X_test': X_test_scaled,
        'y_test': y_test_raw,
        'scaler': scaler
    }

    print(f"[{season.capitalize():<12}] Loaded '{file_path}' | Train: {X_train_res.shape[0]} samples (SMOTE) | Test: {X_test_scaled.shape[0]} samples")

print("\nData preparation complete.\n")

--- [CELL 1] Loading and Preprocessing Seasonal Datasets ---
[Winter      ] Loaded 'data_winter.csv' | Train: 17775 samples (SMOTE) | Test: 2529 samples
[Summer      ] Loaded 'data_summer.csv' | Train: 26604 samples (SMOTE) | Test: 2928 samples
[Monsoon     ] Loaded 'data_monsoon.csv' | Train: 24753 samples (SMOTE) | Test: 3175 samples
[Post_monsoon] Loaded 'data_post_monsoon.csv' | Train: 13272 samples (SMOTE) | Test: 1560 samples

Data preparation complete.



In [ ]:
# ==========================================
# CELL 2: Model Training & Epoch Progress Logging
# ==========================================
from sklearn.linear_model import LogisticRegression

trained_models = {}

print("--- [CELL 2] Training Logistic Regression Models Across Seasons ---")

EPOCHS_STEP = 100
MAX_EPOCHS = 1000

for season in seasons:
    print(f"\nTraining Model for Season: [{season.upper()}]")
    X_tr = processed_data[season]['X_train']
    y_tr = processed_data[season]['y_train']

    # Initialize multinomial logistic regression with SAGA solver for incremental epochs
    model = LogisticRegression(
        multi_class='multinomial',
        solver='saga',
        warm_start=True,
        random_state=42,
        max_iter=EPOCHS_STEP
    )

    # Incremental training logging every 100 epochs
    for current_max in range(EPOCHS_STEP, MAX_EPOCHS + 1, EPOCHS_STEP):
        model.max_iter = current_max
        model.fit(X_tr, y_tr)

        # Calculate training loss (log loss approximation)
        y_proba = model.predict_proba(X_tr)
        # Compute mean negative log likelihood
        classes = model.classes_
        y_onehot = pd.get_dummies(y_tr)[classes].values
        loss = -np.mean(np.sum(y_onehot * np.log(y_proba + 1e-15), axis=1))

        print(f"  Epoch [{current_max:4d}/{MAX_EPOCHS}] - Loss: {loss:.4f} - Converged Iterations: {model.n_iter_[0]}")

        # Stop early if solver converged before reaching max_iter step
        if model.n_iter_[0] < current_max:
            print(f"  --> Solver converged early at iteration {model.n_iter_[0]}. Stopping epoch loop.")
            break

    trained_models[season] = model

print("\nModel training phase complete.\n")

--- [CELL 2] Training Logistic Regression Models Across Seasons ---

Training Model for Season: [WINTER]
  Epoch [ 100/1000] - Loss: 0.4961 - Converged Iterations: 100
  Epoch [ 200/1000] - Loss: 0.4929 - Converged Iterations: 200
  Epoch [ 300/1000] - Loss: 0.4907 - Converged Iterations: 300
  Epoch [ 400/1000] - Loss: 0.4891 - Converged Iterations: 400
  Epoch [ 500/1000] - Loss: 0.4881 - Converged Iterations: 500
  Epoch [ 600/1000] - Loss: 0.4874 - Converged Iterations: 600
  Epoch [ 700/1000] - Loss: 0.4869 - Converged Iterations: 700
  Epoch [ 800/1000] - Loss: 0.4868 - Converged Iterations: 196
  --> Solver converged early at iteration 196. Stopping epoch loop.

Training Model for Season: [SUMMER]
  Epoch [ 100/1000] - Loss: 0.6764 - Converged Iterations: 100
  Epoch [ 200/1000] - Loss: 0.6762 - Converged Iterations: 200
  Epoch [ 300/1000] - Loss: 0.6762 - Converged Iterations: 86
  --> Solver converged early at iteration 86. Stopping epoch loop.

Training Model for Season: [MO

In [ ]:
# ==========================================
# CELL 3: Model Evaluation & Automated CSV Export
# ==========================================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

def calc_gmean(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    recalls = [report[cls]['recall'] for cls in report if cls not in ['accuracy', 'macro avg', 'weighted avg']]
    return np.exp(np.mean(np.log(np.maximum(recalls, 1e-5))))

results = []
detailed_reports = []
evaluation_logs = {}

print("--- [CELL 3] Evaluating Within-Season Test Performance ---")

for season in seasons:
    model = trained_models[season]
    X_te = processed_data[season]['X_test']
    y_te = processed_data[season]['y_test']

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)

    # Calculate Core Metrics
    macro_f1 = f1_score(y_te, y_pred, average='macro')
    g_mean = calc_gmean(y_te, y_pred)

    try:
        macro_auc = roc_auc_score(y_te, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        macro_auc = np.nan

    cm = confusion_matrix(y_te, y_pred)
    report_dict = classification_report(y_te, y_pred, output_dict=True)

    evaluation_logs[season] = {
        'y_true': y_te,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'cm': cm,
        'report': classification_report(y_te, y_pred)
    }

    # Store high-level metrics
    results.append({
        'Model': 'Logistic Regression',
        'Season': season.capitalize(),
        'Macro F1': round(macro_f1, 4),
        'Macro AUC': round(macro_auc, 4),
        'G-Mean': round(g_mean, 4)
    })

    # Store detailed per-class metrics
    for cls_name, metrics in report_dict.items():
        if isinstance(metrics, dict):
            detailed_reports.append({
                'Season': season.capitalize(),
                'Class': cls_name,
                'Precision': round(metrics['precision'], 4),
                'Recall': round(metrics['recall'], 4),
                'F1-Score': round(metrics['f1-score'], 4),
                'Support': int(metrics['support'])
            })

# Save Tables to results/tables/
tables_dir = os.path.join("results", "tables")
os.makedirs(tables_dir, exist_ok=True)

# 1. Export High-Level Summary Table
results_df = pd.DataFrame(results)
summary_csv_path = os.path.join(tables_dir, "logistic_regression_e1.csv")
results_df.to_csv(summary_csv_path, index=False)

# 2. Export Detailed Classification Table
detailed_df = pd.DataFrame(detailed_reports)
detailed_csv_path = os.path.join(tables_dir, "logistic_regression_e1_detailed.csv")
detailed_df.to_csv(detailed_csv_path, index=False)

print("\n=== Logistic Regression: Within-Season Baseline (E1) ===")
print(results_df.to_string(index=False))

print(f"\n[SUCCESS] Exported summary table to: {summary_csv_path}")
print(f"[SUCCESS] Exported detailed classification table to: {detailed_csv_path}")

--- [CELL 3] Evaluating Within-Season Test Performance ---

=== Logistic Regression: Within-Season Baseline (E1) ===
              Model       Season  Macro F1  Macro AUC  G-Mean
Logistic Regression       Winter    0.5806     0.8619  0.7504
Logistic Regression       Summer    0.5611     0.8638  0.7229
Logistic Regression      Monsoon    0.3993     0.6843  0.0154
Logistic Regression Post_monsoon    0.6501     0.9283  0.8270

[SUCCESS] Exported summary table to: results/tables/logistic_regression_e1.csv
[SUCCESS] Exported detailed classification table to: results/tables/logistic_regression_e1_detailed.csv


In [ ]:
# ==========================================
# CELL 4: Individual Slide Graphs & Auto-Download
# ==========================================
import os
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files  # Colab direct file download helper

print("--- [CELL 4] Generating Individual Slide Figures & Downloading ---")

output_dir = os.path.join("results", "visualizations", "logistic_regression")
os.makedirs(output_dir, exist_ok=True)

download_queue = []

# 1. Plot & Save Individual Confusion Matrices for each season
for season in seasons:
    plt.figure(figsize=(6, 5))
    cm = evaluation_logs[season]['cm']
    labels = sorted(list(set(evaluation_logs[season]['y_true'])))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
        annot_kws={"size": 12, "weight": "bold"}
    )
    plt.title(f'Confusion Matrix: {season.capitalize()}', fontsize=14, fontweight='bold', pad=12)
    plt.xlabel('Predicted Tier', fontsize=11, fontweight='bold')
    plt.ylabel('True Tier', fontsize=11, fontweight='bold')
    plt.tight_layout()

    # Save individual file
    file_path = os.path.join(output_dir, f"cm_{season}.png")
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.show()
    download_queue.append(file_path)

# 2. Plot & Save Standalone Feature Importance Graphic (Winter)
plt.figure(figsize=(8, 5))

winter_model = trained_models['winter']
avg_abs_coefs = np.mean(np.abs(winter_model.coef_), axis=0)
top_idx = np.argsort(avg_abs_coefs)[-10:]

bars = plt.barh(np.array(predictors)[top_idx], avg_abs_coefs[top_idx], color='#2b5c8f', edgecolor='black')
plt.title('Top 10 Predictor Importance (Logistic Regression - Winter)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Mean Absolute Coefficient Value', fontsize=11, fontweight='bold')
plt.ylabel('Predictor Feature', fontsize=11, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.3f}',
             va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()

feat_file_path = os.path.join(output_dir, "feature_importance_winter.png")
plt.savefig(feat_file_path, dpi=300, bbox_inches='tight')
plt.show()
download_queue.append(feat_file_path)

# 3. Trigger Browser Downloads for all created graphics
print("\n--- Triggering Automatic Browser Downloads ---")
for file_path in download_queue:
    print(f"Downloading: {file_path}")
    files.download(file_path)